# Laminate pre-design based on Tsai-Hill and Yamada-Sun failure criteria
#### **(by Norbert Blanco)**

This notebook computes and plots the value of the failure index of the Tsai-Hill and Yamada-Sun criteria on a composite plate with UD plies subjected to both in-plane and out-of-plane loading.

Converted from the original MATLAB Live Script (`laminate_predesign_script.m`) to Python. It runs in Jupyter and in Google Colab without any additional setup (only `numpy`, `pandas` and `matplotlib`, all preinstalled in Colab).

## Initialisation

Import the packages used throughout the notebook. The helpers `fmt4` and `fmt_sci` are only used to display the results: `fmt4` writes a number with 4 significant figures (blank space as thousands separator and dot for decimals) and `fmt_sci` writes it in scientific notation with 4 significant figures.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

np.set_printoptions(precision=4, suppress=False)


def fmt4(v, tiny=1e-9):
    """4 significant figures in fixed notation, blank space as thousands separator."""
    v = float(v)
    if np.isinf(v):
        return "inf"
    if abs(v) < tiny:
        return "0"
    decimals = max(0, 3 - int(np.floor(np.log10(abs(v)))))
    return f"{v:,.{decimals}f}".replace(",", " ")


def fmt_sci(v, tiny=1e-15):
    """Scientific notation, 4 significant figures."""
    v = 0.0 if abs(v) < tiny else float(v)
    return f"{v:.3e}"


## Definition of material, LSS and applied loads

#### Definition of material properties

- The material is assumed to be transversely isotropic and only 4 elastic constants are required: E11 (MPa), E22 (MPa), nu12, G12 (MPa).
- The plies are assumed to be working under plane-stress condition and 5 strength properties are required (all in absolute value): X11t (MPa), X11c (MPa), X22t (MPa), X22c (MPa) and X12 (MPa).
- CFRP T300/5208 is defined as material 1. CFRP T300/914 is defined as material 2. Additional materials can be defined following the same structure `Mat[mat_num] = [E11, E22, nu12, G12, X11t, X11c, X22t, X22c, X12]`, or by updating the material properties.

In [ ]:
# Mat[mat_num] = [E11, E22, nu12, G12, X11t, X11c, X22t, X22c, X12]
Mat = {
    1: [181000, 10300, 0.28, 7170, 1500, 1500, 40, 246, 68],
    2: [134000, 10000, 0.25, 4200, 1500, 1400, 50, 200, 75],
}


#### Definition of the Laminate Stacking Sequence

- The LSS is defined through material (previously defined), orientation (in degrees) and thickness (in mm).
- The first layer is assumed to be the one on the bottom, according to the Z direction.
- Define as many layers as required following this structure: `L[layer_num] = [mat_num, orientation, thickness]`.
- If the laminate is symmetric, set `LSS_symm = True` and only define the first half of the LSS. For LSS with odd symmetry, define the central ply with half of the thickness.

In [ ]:
LSS_symm = False

# L[layer_num] = [mat_num, orientation (deg), thickness (mm)]
L = np.array([
    [1,   0, 0.125],
    [1,  45, 0.125],
    [1, -45, 0.125],
    [1,  90, 0.125],
])


#### Definition of the applied unit forces and moments

Define the applied unit forces (N/mm) and unit moments (N) following this structure: `NM = [Nxx, Nyy, Nxy, Mxx, Myy, Mxy]`:

In [ ]:
NM = np.array([50, 0, 0, 0, 0, 0], dtype=float)


## Calculations

First it is necessary to complete the LSS in case a symmetric LSS is defined:

In [ ]:
if LSS_symm:
    L = np.vstack([L, L[::-1]])

n_lam = L.shape[0]
L


#### Calculation and assembly of stiffness, compliance and transformation matrices

The plane-stress compliance matrix S is calculated for each ply according to the definition for a transversely isotropic material. The plane-stress stiffness matrix Q is obtained by inverting S:

$$
S = \begin{bmatrix}
1/E_{11} & -\nu_{12}/E_{11} & 0 \\
-\nu_{12}/E_{11} & 1/E_{22} & 0 \\
0 & 0 & 1/G_{12}
\end{bmatrix}
$$

In [ ]:
S = np.zeros((3, 3, n_lam))
Q = np.zeros((3, 3, n_lam))

for i in range(n_lam):
    i_mat = int(L[i, 0])
    E11, E22, nu12, G12 = Mat[i_mat][:4]

    S[0, 0, i] = 1 / E11
    S[0, 1, i] = -nu12 / E11
    S[1, 0, i] = S[0, 1, i]
    S[1, 1, i] = 1 / E22
    S[2, 2, i] = 1 / G12

    Q[:, :, i] = np.linalg.inv(S[:, :, i])


The transformation matrix T is calculated for each ply according to the definition. The transformation matrix for strain, `Tg`, is also obtained according to the definition:

$$
T = \begin{bmatrix}
m^2 & n^2 & 2mn \\
n^2 & m^2 & -2mn \\
-mn & mn & m^2 - n^2
\end{bmatrix}
\qquad
T_\gamma = \left( T^{-1} \right)^T
\qquad
m = \cos\theta, \; n = \sin\theta
$$

In [ ]:
T = np.zeros((3, 3, n_lam))
Tg = np.zeros((3, 3, n_lam))

for i in range(n_lam):
    theta = L[i, 1] * np.pi / 180
    m = np.cos(theta)
    n = np.sin(theta)

    T[0, 0, i] = m**2
    T[0, 1, i] = n**2
    T[0, 2, i] = 2 * m * n
    T[1, 0, i] = n**2
    T[1, 1, i] = m**2
    T[1, 2, i] = -2 * m * n
    T[2, 0, i] = -m * n
    T[2, 1, i] = m * n
    T[2, 2, i] = m**2 - n**2

    Tg[:, :, i] = np.linalg.inv(T[:, :, i]).T


The transformed compliance and stiffness matrices are obtained for each ply:

$$
[\bar{S}] = [T_\gamma]^{-1} [S] [T]
\qquad
[\bar{Q}] = [\bar{S}]^{-1}
$$

In [ ]:
Sb = np.zeros((3, 3, n_lam))
Qb = np.zeros((3, 3, n_lam))

for i in range(n_lam):
    Sb[:, :, i] = np.linalg.inv(Tg[:, :, i]) @ S[:, :, i] @ T[:, :, i]
    Qb[:, :, i] = np.linalg.inv(Sb[:, :, i])


### Calculation of the laminate's constitutive equation

Before determining matrices A, B and D, it is necessary to determine the total thickness of the laminate and the coordinates of each ply surface from the laminate's mid-plane:

In [ ]:
TH = L[:, 2].sum()
th = TH / 2

Z = np.zeros(n_lam + 1)
Z[0] = -th
running = 0.0
for i in range(n_lam):
    running += L[i, 2]
    Z[i + 1] = running - th

Z


#### Calculation of matrices A, B and D

Matrices A, B and D are calculated taking into account the bottom and top coordinates of the layers and the corresponding stiffness matrices:

$$
A_{jk} = \sum_{i=1}^{n} \bar{Q}_{jk,i} (Z_{i+1} - Z_i) \qquad
B_{jk} = \sum_{i=1}^{n} \bar{Q}_{jk,i} \frac{Z_{i+1}^2 - Z_i^2}{2} \qquad
D_{jk} = \sum_{i=1}^{n} \bar{Q}_{jk,i} \frac{Z_{i+1}^3 - Z_i^3}{3}
$$

In [ ]:
A = np.zeros((3, 3))
B = np.zeros((3, 3))
D = np.zeros((3, 3))

for i in range(n_lam):
    A += Qb[:, :, i] * (Z[i + 1] - Z[i])
    B += Qb[:, :, i] * (Z[i + 1]**2 - Z[i]**2) / 2
    D += Qb[:, :, i] * (Z[i + 1]**3 - Z[i]**3) / 3

ABD = np.block([[A, B], [B, D]])


#### Calculation of the laminate's strains and stresses

First it is necessary to determine the laminate's mid-plane strains and curvatures by inverting matrix ABD and multiplying by the unit forces and moments vector:

$$
\begin{bmatrix} \varepsilon^0 \\ \kappa \end{bmatrix} =
\begin{bmatrix} A & B \\ B & D \end{bmatrix}^{-1}
\begin{bmatrix} N \\ M \end{bmatrix}
$$

In [ ]:
eps_kappa = np.linalg.inv(ABD) @ NM


Strains and stresses at the bottom and top locations of each ply are calculated taking into account the laminate's mid-plane strains and curvatures and the locations of each ply (columns of each array: bottom, top). The stresses in the local axes of the ply are obtained from the strains in the local axes, and then they are rotated to the global axes:

$$
[\varepsilon]^k = [\varepsilon^0] + z^k [\kappa]
\qquad
[\sigma]^k = [\bar{Q}]^k [\varepsilon^0] + z^k [\bar{Q}]^k [\kappa]
$$

In [ ]:
eps_xyz = np.zeros((3, 2, n_lam))
eps_123 = np.zeros((3, 2, n_lam))
sigma_xyz = np.zeros((3, 2, n_lam))
sigma_123 = np.zeros((3, 2, n_lam))

for i in range(n_lam):
    eps_xyz[:, 0, i] = eps_kappa[0:3] + Z[i] * eps_kappa[3:6]
    eps_xyz[:, 1, i] = eps_kappa[0:3] + Z[i + 1] * eps_kappa[3:6]
    eps_123[:, :, i] = Tg[:, :, i] @ eps_xyz[:, :, i]
    sigma_123[:, :, i] = Q[:, :, i] @ eps_123[:, :, i]
    sigma_xyz[:, :, i] = np.linalg.inv(T[:, :, i]) @ sigma_123[:, :, i]


#### Calculation of the Failure Index according to the Tsai-Hill failure criterion

The failure index is calculated according to the 2D form of the Tsai-Hill and the Yamada-Sun criteria, respectively, for each ply at bottom and top positions as:

$$
FI_{TH} = \left(\frac{\sigma_{11}}{\sigma^u_{11T/C}}\right)^2 + \left(\frac{\sigma_{22}}{\sigma^u_{22T/C}}\right)^2 - \frac{\sigma_{11}\,\sigma_{22}}{\left(\sigma^u_{11T/C}\right)^2} + \left(\frac{\sigma_{12}}{\sigma^u_{12}}\right)^2
$$

$$
FI_{YS} = \left(\frac{\sigma_{11}}{\sigma^u_{11T/C}}\right)^2 + \left(\frac{\sigma_{12}}{\sigma^u_{12}}\right)^2
$$

Failure takes place when the index reaches 1. Although the Tsai-Hill and the Yamada-Sun criteria do not consider different strengths for tension or compression, the tensile and compressive stresses in the longitudinal and transverse directions will be compared with the respective tensile and compression strengths.

In [ ]:
TH_A = np.zeros((2, n_lam))
TH_B = np.zeros((2, n_lam))
TH_C = np.zeros((2, n_lam))
TH_D = np.zeros((2, n_lam))
fi_TH = np.zeros((2, n_lam))
fi_YS = np.zeros((2, n_lam))

for i in range(n_lam):
    X11t, X11c, X22t, X22c, X12 = Mat[int(L[i, 0])][4:9]
    for j in range(2):
        s11, s22, s12 = sigma_123[:, j, i]
        sigma_11u = X11t if s11 >= 0 else X11c
        sigma_22u = X22t if s22 >= 0 else X22c

        TH_A[j, i] = s11**2 / sigma_11u**2
        TH_B[j, i] = s22**2 / sigma_22u**2
        TH_C[j, i] = -s11 * s22 / sigma_11u**2
        TH_D[j, i] = s12**2 / X12**2
        fi_TH[j, i] = TH_A[j, i] + TH_B[j, i] + TH_C[j, i] + TH_D[j, i]
        fi_YS[j, i] = TH_A[j, i] + TH_D[j, i]


In addition to the failure index, the inverse reserve factor and the reserve factor can be also calculated for the Tsai-Hill and Yamada-Sun criteria, respectively. Both failure indices are quadratic in stress, so scaling the load by RF multiplies FI by RF$^2$ and failure takes place when $RF^2 \cdot FI = 1$:

$$
IRF = \sqrt{FI} \qquad RF = \frac{1}{IRF}
$$

In [ ]:
IRF_TH = fi_TH ** 0.5
IRF_YS = fi_YS ** 0.5
with np.errstate(divide="ignore"):
    RF_TH = 1 / IRF_TH
    RF_YS = 1 / IRF_YS


## Results

In the tables below, the layers are listed as they are stacked in the laminate (the last layer on top and the first layer at the bottom). For each layer, the values at its top surface (Top) are shown above the values at its bottom surface (Bot). The first group of columns corresponds to the local axes of the ply (11, 22, 12) and the second group to the global axes (xx, yy, xy).

#### Local and global strain

The resulting distribution of strain in local and global coordinates at the bottom and top positions of each ply along the laminate thickness is:

In [ ]:
#@title { display-mode: "form" }
# Rows ordered from the top of the laminate (last layer, Top) to the bottom (first layer, Bot)
ZZ = np.zeros(2 * n_lam)
layer = [0] * (2 * n_lam)
pos = [''] * (2 * n_lam)
eps_loc = np.zeros((2 * n_lam, 3))
eps_glob = np.zeros((2 * n_lam, 3))
sigma_loc = np.zeros((2 * n_lam, 3))
sigma_glob = np.zeros((2 * n_lam, 3))
fi_plot_TH = np.zeros(2 * n_lam)
fi_plot_YS = np.zeros(2 * n_lam)

for i in range(n_lam):
    r_top = 2 * (n_lam - 1 - i)
    r_bot = r_top + 1
    ZZ[r_top], ZZ[r_bot] = Z[i + 1], Z[i]
    layer[r_top], layer[r_bot] = i + 1, i + 1
    pos[r_top], pos[r_bot] = "Top", "Bot"
    for r, j in ((r_top, 1), (r_bot, 0)):
        eps_loc[r] = eps_123[:, j, i]
        eps_glob[r] = eps_xyz[:, j, i]
        sigma_loc[r] = sigma_123[:, j, i]
        sigma_glob[r] = sigma_xyz[:, j, i]
        fi_plot_TH[r] = fi_TH[j, i]
        fi_plot_YS[r] = fi_YS[j, i]

IRF_plot_TH, IRF_plot_YS = fi_plot_TH ** 0.5, fi_plot_YS ** 0.5
with np.errstate(divide="ignore"):
    RF_plot_TH, RF_plot_YS = 1 / IRF_plot_TH, 1 / IRF_plot_YS

index = pd.MultiIndex.from_arrays([layer, pos], names=["Layer", "Position"])


def result_table(loc, glob, formatter, local_labels, global_labels):
    columns = pd.MultiIndex.from_tuples(
        [("Local axes", c) for c in local_labels] + [("Global axes", c) for c in global_labels])
    data = [[formatter(v) for v in a] + [formatter(v) for v in b] for a, b in zip(loc, glob)]
    return pd.DataFrame(data, index=index, columns=columns)


print("Strain [-]")
display(result_table(eps_loc, eps_glob, fmt_sci, ["e11", "e22", "e12"], ["exx", "eyy", "exy"]))


In [ ]:
#@title { display-mode: "form" }
def strain_plot(data, labels):
    fig, ax = plt.subplots()
    for k, label in enumerate(labels):
        ax.plot(data[:, k], ZZ, marker='o', label=label)
    ax.set_xlabel('Strain (m/m)')
    ax.set_ylabel('Laminate thickness (mm)')
    ax.legend()
    ax.grid(True)
    plt.show()


strain_plot(eps_loc, [r'$\varepsilon_{11}$', r'$\varepsilon_{22}$', r'$\varepsilon_{12}$'])
strain_plot(eps_glob, [r'$\varepsilon_{xx}$', r'$\varepsilon_{yy}$', r'$\varepsilon_{xy}$'])


#### Local and global stress

The resulting distribution of stress in local and global coordinates at the bottom and top positions of each ply along the laminate thickness is:

In [ ]:
#@title { display-mode: "form" }
print("Stress [MPa]")
display(result_table(sigma_loc, sigma_glob, fmt4, ["s11", "s22", "s12"], ["sxx", "syy", "sxy"]))


In [ ]:
#@title { display-mode: "form" }
def stress_plot(data, labels):
    fig, ax = plt.subplots()
    for k, label in enumerate(labels):
        ax.plot(data[:, k], ZZ, marker='o', label=label)
    ax.set_xlabel('Stress (MPa)')
    ax.set_ylabel('Laminate thickness (mm)')
    ax.legend()
    ax.grid(True)
    plt.show()


stress_plot(sigma_loc, [r'$\sigma_{11}$', r'$\sigma_{22}$', r'$\sigma_{12}$'])
stress_plot(sigma_glob, [r'$\sigma_{xx}$', r'$\sigma_{yy}$', r'$\sigma_{xy}$'])


#### Failure index

The resulting distribution of failure index, inverse reserve factor and reserve factor at the bottom and top positions of each ply along the laminate thickness is:

In [ ]:
#@title { display-mode: "form" }
names = ['FI-TH', 'IRF-TH', 'RF-TH', 'FI-YS', 'IRF-YS', 'RF-YS']
data = [[fmt4(v) for v in row] for row in np.column_stack(
    [fi_plot_TH, IRF_plot_TH, RF_plot_TH, fi_plot_YS, IRF_plot_YS, RF_plot_YS])]
FI_res = pd.DataFrame(data, index=index, columns=names)
FI_res


In [ ]:
#@title { display-mode: "form" }
def failure_plot(data_TH, data_YS, xlabel):
    fig, ax = plt.subplots()
    ax.plot(data_TH, ZZ, marker='o', label='Tsai-Hill')
    ax.plot(data_YS, ZZ, marker='o', label='Yamada-Sun')
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Laminate thickness (mm)')
    ax.legend()
    ax.grid(True)
    plt.show()


failure_plot(fi_plot_TH, fi_plot_YS, 'Failure Index')
failure_plot(IRF_plot_TH, IRF_plot_YS, 'Inverse Reserve Factor')
failure_plot(RF_plot_TH, RF_plot_YS, 'Reserve Factor')
